# Sample current clips

This notebook reads the clips that are currently available under `data/clips/`, joins label metadata from `data/labels.jsonl` when present, and samples 100 clips for inspection.


In [1]:
import shutil
from pathlib import Path

import pandas as pd
from IPython.display import Video, display

from aitraf.data_ops.schema import LabelsSchema
from aitraf.data_ops.utils import strip_clips_prefix

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
CLIPS_DIR = DATA_DIR / "clips"
SAMPLED_CLIPS_DIR = DATA_DIR / "clips_sampled"
LABELS_PATH = DATA_DIR / "labels.jsonl"

SAMPLE_SIZE = 100
RANDOM_SEED = 7
PREVIEW_COUNT = 5

pd.set_option("display.max_colwidth", 120)


In [2]:
def s3_uri_to_clip_name(video_uri: str) -> str:
    key = video_uri.replace("s3://aitraf/", "", 1)
    return strip_clips_prefix(Path(key)).name


clip_paths = sorted(CLIPS_DIR.rglob("*.mp4"))
if not clip_paths:
    raise FileNotFoundError(f"No clips found under {CLIPS_DIR}")

clips_df = pd.DataFrame({"clip_path": clip_paths})
clips_df["clip_name"] = clips_df["clip_path"].map(lambda path: path.name)
clips_df["clip_relpath"] = clips_df["clip_path"].map(
    lambda path: path.relative_to(PROJECT_ROOT).as_posix()
)

if LABELS_PATH.exists():
    labels_df = pd.read_json(LABELS_PATH, lines=True, dtype=LabelsSchema.types)
    labels_df["clip_name"] = labels_df["video"].map(s3_uri_to_clip_name)
    labels_df = labels_df[
        [
            "clip_name",
            "video",
            "trick",
            "person",
            "key_foot",
            "execution_score",
        ]
    ]
    current_clips_df = clips_df.merge(labels_df, on="clip_name", how="left")
else:
    current_clips_df = clips_df.assign(
        video=pd.NA,
        trick=pd.NA,
        person=pd.NA,
        key_foot=pd.NA,
        execution_score=pd.NA,
    )

unlabeled_clips_df = current_clips_df[current_clips_df["trick"].isna()].copy()

deleted_count = 0
for clip_path in unlabeled_clips_df["clip_path"]:
    if clip_path.exists():
        clip_path.unlink()
        deleted_count += 1
current_clips_df = current_clips_df[current_clips_df["trick"].notna()].reset_index(drop=True)
print(f"Deleted {deleted_count} unlabeled clips from {CLIPS_DIR}")

sample_n = min(SAMPLE_SIZE, len(current_clips_df))
sampled_df = (
    current_clips_df.sample(n=sample_n, random_state=RANDOM_SEED)
    .sort_values(["trick", "person", "clip_name"], na_position="last")
    .reset_index(drop=True)
)

matched_labels = int(current_clips_df["trick"].notna().sum())
copied_count = 0
sampled_copy_paths = []
for row in sampled_df.itertuples(index=False):
    destination_path = SAMPLED_CLIPS_DIR / row.clip_path.relative_to(CLIPS_DIR)
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(row.clip_path, destination_path)
    sampled_copy_paths.append(destination_path)
    copied_count += 1

sampled_df["sampled_clip_path"] = sampled_copy_paths
sampled_df["sampled_clip_relpath"] = sampled_df["sampled_clip_path"].map(
    lambda path: path.relative_to(PROJECT_ROOT).as_posix()
)

print(f"Current clips found: {len(current_clips_df)}")
print(f"Clips matched with labels: {matched_labels}")
print(f"Sampling {sample_n} clips with random seed {RANDOM_SEED}")
print(f"Copied {copied_count} sampled clips to {SAMPLED_CLIPS_DIR}")

sampled_df.head()


Deleted 0 unlabeled clips from /workspace/data/clips
Current clips found: 689
Clips matched with labels: 689
Sampling 100 clips with random seed 7
Copied 100 sampled clips to /workspace/data/clips_sampled


clip_path  \
0  /workspace/data/clips/25-11-07 15-47-00 5699-00.07.46.737-00.07.52.252-seg14.mp4   
1  /workspace/data/clips/25-11-15 18-42-16 5731-00.18.16.571-00.18.22.097-seg34.mp4   
2  /workspace/data/clips/25-11-15 19-11-43 5734-00.15.03.295-00.15.06.875-seg27.mp4   
3  /workspace/data/clips/25-11-17 21-01-19 5741-00.13.33.598-00.13.37.378-seg35.mp4   
4  /workspace/data/clips/25-11-17 21-01-19 5741-00.23.15.487-00.23.20.437-seg52.mp4   

                                                    clip_name  \
0  25-11-07 15-47-00 5699-00.07.46.737-00.07.52.252-seg14.mp4   
1  25-11-15 18-42-16 5731-00.18.16.571-00.18.22.097-seg34.mp4   
2  25-11-15 19-11-43 5734-00.15.03.295-00.15.06.875-seg27.mp4   
3  25-11-17 21-01-19 5741-00.13.33.598-00.13.37.378-seg35.mp4   
4  25-11-17 21-01-19 5741-00.23.15.487-00.23.20.437-seg52.mp4   

                                                            clip_relpath  \
0  data/clips/25-11-07 15-47-00 5699-00.07.46.737-00.07.52.252-seg14.mp4   
1  data/clips/25-11-15 18-42-16 5731-00.18.16.571-00.18.22.097-seg34.mp4   
2  data/clips/25-11-15 19-11-43 5734-00.15.03.295-00.15.06.875-seg27.mp4   
3  data/clips/25-11-17 21-01-19 5741-00.13.33.598-00.13.37.378-seg35.mp4   
4  data/clips/25-11-17 21-01-19 5741-00.23.15.487-00.23.20.437-seg52.mp4   

                                                                          video  \
0  s3://aitraf/clips/25-11-07 15-47-00 5699-00.07.46.737-00.07.52.252-seg14.mp4   
1  s3://aitraf/clips/25-11-15 18-42-16 5731-00.18.16.571-00.18.22.097-seg34.mp4   
2  s3://aitraf/clips/25-11-15 19-11-43 5734-00.15.03.295-00.15.06.875-seg27.mp4   
3  s3://aitraf/clips/25-11-17 21-01-19 5741-00.13.33.598-00.13.37.378-seg35.mp4   
4  s3://aitraf/clips/25-11-17 21-01-19 5741-00.23.15.487-00.23.20.437-seg52.mp4   

     trick    person key_foot  execution_score  \
0  ao-soul  Henrikas    right                3   
1  ao-soul  Henrikas    right                3   
2  ao-soul  Henrikas    right                1   
3  ao-soul  Henrikas     left                2   
4  ao-soul  Henrikas    right                3   

0                                                 Not perfect, but stylish overall   
1                                        A bit unstable balance. But great length.   
2                                                                       not landed   
3  Very sketchy landing. And short length too, but the body movements looks right.   
4                Looks controlled, but on the slower side and just a bit too short   

                                                                          sampled_clip_path  \
0  /workspace/data/clips_sampled/25-11-07 15-47-00 5699-00.07.46.737-00.07.52.252-seg14.mp4   
1  /workspace/data/clips_sampled/25-11-15 18-42-16 5731-00.18.16.571-00.18.22.097-seg34.mp4   
2  /workspace/data/clips_sampled/25-11-15 19-11-43 5734-00.15.03.295-00.15.06.875-seg27.mp4   
3  /workspace/data/clips_sampled/25-11-17 21-01-19 5741-00.13.33.598-00.13.37.378-seg35.mp4   
4  /workspace/data/clips_sampled/25-11-17 21-01-19 5741-00.23.15.487-00.23.20.437-seg52.mp4   

                                                            sampled_clip_relpath  
0  data/clips_sampled/25-11-07 15-47-00 5699-00.07.46.737-00.07.52.252-seg14.mp4  
1  data/clips_sampled/25-11-15 18-42-16 5731-00.18.16.571-00.18.22.097-seg34.mp4  
2  data/clips_sampled/25-11-15 19-11-43 5734-00.15.03.295-00.15.06.875-seg27.mp4  
3  data/clips_sampled/25-11-17 21-01-19 5741-00.13.33.598-00.13.37.378-seg35.mp4  
4  data/clips_sampled/25-11-17 21-01-19 5741-00.23.15.487-00.23.20.437-seg52.mp4

In [3]:
summary_df = pd.DataFrame(
    [
        {"metric": "current_clips", "value": len(current_clips_df)},
        {
            "metric": "clips_with_labels",
            "value": int(current_clips_df["trick"].notna().sum()),
        },
        {
            "metric": "clips_without_labels",
            "value": 0,
        },
        {"metric": "deleted_unlabeled_clips", "value": deleted_count},
        {"metric": "copied_sampled_clips", "value": copied_count},
        {"metric": "sample_size", "value": len(sampled_df)},
    ]
)

sample_trick_counts = (
    sampled_df["trick"].fillna("unlabeled").value_counts().rename_axis("trick").to_frame("sample_count")
)

display(summary_df)
display(sample_trick_counts)
display(
    sampled_df[
        [
            "clip_name",
            "clip_relpath",
            "sampled_clip_relpath",
            "trick",
            "person",
            "key_foot",
            "execution_score",
        ]
    ]
)


,metric,value
0,current_clips,689
1,clips_with_labels,689
2,clips_without_labels,0
3,deleted_unlabeled_clips,0
4,copied_sampled_clips,100
5,sample_size,100


,sample_count
trick,
fs-savanah,18
ao-soul,16
soul,15
mizou,14
bs-royale,14
top-soul,12
fs-royale,11


clip_name  \
0   25-11-07 15-47-00 5699-00.07.46.737-00.07.52.252-seg14.mp4   
1   25-11-15 18-42-16 5731-00.18.16.571-00.18.22.097-seg34.mp4   
2   25-11-15 19-11-43 5734-00.15.03.295-00.15.06.875-seg27.mp4   
3   25-11-17 21-01-19 5741-00.13.33.598-00.13.37.378-seg35.mp4   
4   25-11-17 21-01-19 5741-00.23.15.487-00.23.20.437-seg52.mp4   
..                                                         ...   
95                 IMG_5680-00.02.30.185-00.02.36.031-seg4.mp4   
96  25-11-02 19-14-55 5695-00.01.30.190-00.01.34.805-seg54.mp4   
97  25-10-31 19-46-26 5688-00.04.33.222-00.04.38.667-seg12.mp4   
98  25-12-02 20-41-10 5974-00.10.48.257-00.10.52.854-seg34.mp4   
99   25-12-02 21-20-36 5981-00.01.47.375-00.01.51.245-seg4.mp4   

                                                             clip_relpath  \
0   data/clips/25-11-07 15-47-00 5699-00.07.46.737-00.07.52.252-seg14.mp4   
1   data/clips/25-11-15 18-42-16 5731-00.18.16.571-00.18.22.097-seg34.mp4   
2   data/clips/25-11-15 19-11-43 5734-00.15.03.295-00.15.06.875-seg27.mp4   
3   data/clips/25-11-17 21-01-19 5741-00.13.33.598-00.13.37.378-seg35.mp4   
4   data/clips/25-11-17 21-01-19 5741-00.23.15.487-00.23.20.437-seg52.mp4   
..                                                                    ...   
95                 data/clips/IMG_5680-00.02.30.185-00.02.36.031-seg4.mp4   
96  data/clips/25-11-02 19-14-55 5695-00.01.30.190-00.01.34.805-seg54.mp4   
97  data/clips/25-10-31 19-46-26 5688-00.04.33.222-00.04.38.667-seg12.mp4   
98  data/clips/25-12-02 20-41-10 5974-00.10.48.257-00.10.52.854-seg34.mp4   
99   data/clips/25-12-02 21-20-36 5981-00.01.47.375-00.01.51.245-seg4.mp4   

                                                             sampled_clip_relpath  \
0   data/clips_sampled/25-11-07 15-47-00 5699-00.07.46.737-00.07.52.252-seg14.mp4   
1   data/clips_sampled/25-11-15 18-42-16 5731-00.18.16.571-00.18.22.097-seg34.mp4   
2   data/clips_sampled/25-11-15 19-11-43 5734-00.15.03.295-00.15.06.875-seg27.mp4   
3   data/clips_sampled/25-11-17 21-01-19 5741-00.13.33.598-00.13.37.378-seg35.mp4   
4   data/clips_sampled/25-11-17 21-01-19 5741-00.23.15.487-00.23.20.437-seg52.mp4   
..                                                                            ...   
95                 data/clips_sampled/IMG_5680-00.02.30.185-00.02.36.031-seg4.mp4   
96  data/clips_sampled/25-11-02 19-14-55 5695-00.01.30.190-00.01.34.805-seg54.mp4   
97  data/clips_sampled/25-10-31 19-46-26 5688-00.04.33.222-00.04.38.667-seg12.mp4   
98  data/clips_sampled/25-12-02 20-41-10 5974-00.10.48.257-00.10.52.854-seg34.mp4   
99   data/clips_sampled/25-12-02 21-20-36 5981-00.01.47.375-00.01.51.245-seg4.mp4   

       trick    person key_foot  execution_score  \
0    ao-soul  Henrikas    right                3   
1    ao-soul  Henrikas    right                3   
2    ao-soul  Henrikas    right                1   
3    ao-soul  Henrikas     left                2   
4    ao-soul  Henrikas    right                3   
..       ...       ...      ...              ...   
95  top-soul  Henrikas    right                2   
96  top-soul    Justas    right                1   
97  top-soul       Max    right                3   
98  top-soul    Vilius     left                3   
99  top-soul    Vilius     left                4   

0                                                             Not perfect, but stylish overall  
1                                                    A bit unstable balance. But great length.  
2                                                                                   not landed  
3              Very sketchy landing. And short length too, but the body movements looks right.  
4                            Looks controlled, but on the slower side and just a bit too short  
..                                                                                         ...  
95                   Really low body position, but just way too short. Landings also off a bit  
96        

In [4]:
# Preview a few of the sampled clips inline.
preview_df = sampled_df.head(PREVIEW_COUNT)
display(preview_df[["clip_name", "trick", "person", "execution_score"]])

for row in preview_df.itertuples(index=False):
    print(f"{row.clip_name} | trick={row.trick} | person={row.person} | score={row.execution_score}")
    display(Video(str(row.clip_path), embed=False, width=320))


,clip_name,trick,person,execution_score
0,25-11-07 15-47-00 5699-00.07.46.737-00.07.52.252-seg14.mp4,ao-soul,Henrikas,3
1,25-11-15 18-42-16 5731-00.18.16.571-00.18.22.097-seg34.mp4,ao-soul,Henrikas,3
2,25-11-15 19-11-43 5734-00.15.03.295-00.15.06.875-seg27.mp4,ao-soul,Henrikas,1
3,25-11-17 21-01-19 5741-00.13.33.598-00.13.37.378-seg35.mp4,ao-soul,Henrikas,2
4,25-11-17 21-01-19 5741-00.23.15.487-00.23.20.437-seg52.mp4,ao-soul,Henrikas,3


25-11-07 15-47-00 5699-00.07.46.737-00.07.52.252-seg14.mp4 | trick=ao-soul | person=Henrikas | score=3


25-11-15 18-42-16 5731-00.18.16.571-00.18.22.097-seg34.mp4 | trick=ao-soul | person=Henrikas | score=3


25-11-15 19-11-43 5734-00.15.03.295-00.15.06.875-seg27.mp4 | trick=ao-soul | person=Henrikas | score=1


25-11-17 21-01-19 5741-00.13.33.598-00.13.37.378-seg35.mp4 | trick=ao-soul | person=Henrikas | score=2


25-11-17 21-01-19 5741-00.23.15.487-00.23.20.437-seg52.mp4 | trick=ao-soul | person=Henrikas | score=3
